<a href="https://colab.research.google.com/github/subhasish52/ML_d1_project/blob/main/ML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df=pd.read_csv("/content/AI_health.csv")
df.shape

FileNotFoundError: [Errno 2] No such file or directory: '/content/AI_health.csv'

In [ ]:
cat_cols = ["Brand", "Product_Type", "Category", "Country", "Source_Type"]
encoders = {}

df_enc = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le

In [ ]:
feature_cols = [
    "Brand","Product_Type","Category","Price_INR","Avg_Rating","Num_Reviews",
    "Ingredient_Quality(%)","Clinical_Approval(%)","Eco_Friendliness(%)",
    "User_Trust_Score(%)","Effectiveness_Score(%)","Side_Effect_Rate(%)",
    "Monthly_Sales","Country","Source_Type"
]

X = df_enc[feature_cols]
y = df_enc["Beneficial_Label"]


In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=250,
    max_depth=18,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=18, n_estimators=250, n_jobs=-1,
                       random_state=42)

In [ ]:
y_pred = model.predict(X_test)

print("\nAccuracy:", round(accuracy_score(y_test, y_pred) * 100, 3), "%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 96.04 %

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98     19209
           1       0.33      0.00      0.00       791

    accuracy                           0.96     20000
   macro avg       0.65      0.50      0.49     20000
weighted avg       0.94      0.96      0.94     20000


Confusion Matrix:
 [[19207     2]
 [  790     1]]


In [ ]:
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoders, "encoders.pkl")

['encoders.pkl']

In [ ]:
model = joblib.load("/content/model.pkl")
scaler = joblib.load("/content/scaler.pkl")
encoders = joblib.load("/content/encoders.pkl")

In [ ]:
def analyze_product(input_dict, dataset=df):
    encoded = input_dict.copy()
    for col, encoder in encoders.items():
        try:
            encoded[col] = encoder.transform([encoded[col]])[0]
        except:
            encoded[col] = 0

    feature_cols = [
        "Brand","Product_Type","Category","Price_INR","Avg_Rating","Num_Reviews",
        "Ingredient_Quality(%)","Clinical_Approval(%)","Eco_Friendliness(%)",
        "User_Trust_Score(%)","Effectiveness_Score(%)","Side_Effect_Rate(%)",
        "Monthly_Sales","Country","Source_Type"
    ]

    X = pd.DataFrame([encoded], columns=feature_cols)
    X_scaled = scaler.transform(X)

    pred = model.predict(X_scaled)[0]
    prediction = "Beneficial" if pred == 1 else "Not Beneficial"


    reco_cols = [
        "Price_INR","Avg_Rating","Ingredient_Quality(%)","Clinical_Approval(%)",
        "Eco_Friendliness(%)","User_Trust_Score(%)","Effectiveness_Score(%)",
        "Side_Effect_Rate(%)"
    ]

    reco_df = dataset[reco_cols]
    reco_norm = (reco_df - reco_df.min()) / (reco_df.max() - reco_df.min() + 1e-9)

    user_vec = np.array([
        input_dict["Price_INR"],
        input_dict["Avg_Rating"],
        input_dict["Ingredient_Quality(%)"],
        input_dict["Clinical_Approval(%)"],
        input_dict["Eco_Friendliness(%)"],
        input_dict["User_Trust_Score(%)"],
        input_dict["Effectiveness_Score(%)"],
        input_dict["Side_Effect_Rate(%)"]
    ]).reshape(1,-1)

    user_norm = (user_vec - reco_df.min().values) / (reco_df.max().values - reco_df.min().values + 1e-9)

    sims = np.dot(reco_norm.values, user_norm.T).flatten()
    dataset["similarity"] = sims

    rec = (
        dataset[dataset["Health_Impact_Score"] > input_dict["Health_Score_User"]]
        .sort_values("similarity", ascending=False)
        .head(5)[["Product_ID","Product_Type","Health_Impact_Score","Avg_Rating"]]
        .reset_index(drop=True)
    )

    return prediction, rec


In [ ]:
sample_input = {
    "Brand": "MuscleBlaze",
    "Product_Type": "Protein Powder",
    "Category": "Nutrition",
    "Price_INR": 2499,
    "Avg_Rating": 4.3,
    "Num_Reviews": 1500,
    "Ingredient_Quality(%)": 85,
    "Clinical_Approval(%)": 70,
    "Eco_Friendliness(%)": 60,
    "User_Trust_Score(%)": 78,
    "Effectiveness_Score(%)": 82,
    "Side_Effect_Rate(%)": 4.5,
    "Monthly_Sales": 3800,
    "Country": "India",
    "Source_Type": "AmazonReviews-like",
    "Health_Score_User": 72
}


In [ ]:
prediction, recommendations = analyze_product(sample_input)

print("Prediction:", prediction)
print("\nTop 5 Healthier Alternatives:")
recommendations


Prediction: Not Beneficial

Top 5 Healthier Alternatives:


,Product_ID,Product_Type,Health_Impact_Score,Avg_Rating
0,HP494767,Dumbbell,73.02,4.92
1,HP457482,Multivitamin,80.16,4.74
2,HP434538,Resistance Band,91.31,4.74
3,HP415718,Omega-3 Supplement,73.48,4.93
4,HP411822,Pre-Workout,77.30,4.19
